# Tutorial for CombinedTree and CombinedForest
## 1. Introduction
### 1.1 What is a CombinedTree?
A **CombinedTree** is an individual model whose computation is split into several sub-trees (`A`, `B`, `C`, ...). Each sub-tree is an independent tree evolved by the genetic algorithm, and the sub-tree outputs are glued back together by a fixed, user-written rule. For example, if you want to model:
```python
formula = lambda A, B, C: A + B * C
```
- `A`, `B`, and `C` are separate sub-trees, each evolved independently.
- The final output of the CombinedTree is computed by substituting the outputs of sub-trees `A`, `B`, `C` into the formula (`A + B * C`).
The rule itself is **not** stored inside the tree classes: EvoGP only needs to know which inputs feed each sub-tree (the `data_info` mapping, see section 2.1) and how many sub-trees there are. You write the rule yourself, both in the loss you optimize (section 3) and when you inspect the evolved solution (section 4).

### 1.2 What is a CombinedForest?
A **CombinedForest** is simply a _population_ of CombinedTrees. Since EvoGP is a population-based genetic programming framework, we often manipulate and evaluate populations (forests) as the central objects rather than individual solutions (trees). 
Thus, practically, we usually:
1. Define a **CombinedForest** which internally maintains multiple sub-forests (one for each parameter).
2. Evolve it via the usual genetic operators (selection, crossover, mutation) in EvoGP.


## 2. How to Create a CombinedForest
When creating a **CombinedForest**, there are three important elements to provide:
1. **`data_info`** (which named inputs feed each sub-tree, and the names of the sub-trees themselves).
2. **Descriptors** (the blueprint that controls how each sub-tree is generated and mutated).
3. **Population size** (`pop_size`).
Every sub-tree is single-output (`output_len=1`), and the input names used in `data_info` are also the keys of the dictionaries that `CombinedTree.forward` / `CombinedForest.forward` and `batch_forward` expect.


### 2.1 Defining a `data_info` mapping
`data_info` is a Python `dict` that maps **one output name per sub-tree** to **the list of input names it consumes**:
- every key (`"A"`, `"B"`, `"C"`, ...) is one sub-tree that will be evolved;
- the listed inputs are concatenated, in that order, to form that sub-tree's input vector, so `len(data_info[key])` is the sub-tree's `input_len`;
- different sub-trees may consume different inputs, and the same input name may appear in several sub-trees.
For example, for the rule `A + B * C` over the three variables `x0`, `x1` and `x2`:
```python
data_info = {
    "A": ["x0", "x1", "x2"],
    "B": ["x0", "x1", "x2"],
    "C": ["x0", "x1", "x2"],
}
```
An asymmetric mapping such as `{"A": ["x0"], "B": ["x1", "x2"]}` is equally valid.
The top-level rule is written by you where the model is scored - with the `CustomLoss` problem used in the complete example below, that looks like:
```python
formula = lambda A, B, C: A + B * C
```


### 2.2 Descriptors
**Descriptors** in EvoGP define how trees (individuals) are generated and mutated. Each key of your `data_info` corresponds to one descriptor, and `CombinedForest.random_generate` validates the sizes for you: `descriptors[i].input_len` must equal `len(data_info[key])` and `descriptors[i].output_len` must be `1`.
- **Single Descriptor**: You can pass one `GenerateDescriptor` object if you want _all parameters_ (`A`, `B`, `C`, ...) to be generated with the same rules.
- **List of Descriptors**: If you need different rules for each parameter, supply a list of descriptors, one per `data_info` key.
Check out `evogp_intro.ipynb` or the main EvoGP tutorials for details on `GenerateDescriptor`. Here is a reminder of what it might look like:


In [1]:
from evogp.tree import CombinedForest, GenerateDescriptor
descriptor = GenerateDescriptor(
    max_tree_len=16,
    input_len=3,
    output_len=1,
    using_funcs=["+", "-", "*", "/", "neg"],  # Allowed function set
    max_layer_cnt=4,
    const_samples=[0, 0.5, 1]  # constant choices
)

## 3. Evolving a CombinedForest with an Algorithm
In EvoGP, after you create a **CombinedForest**, you typically feed it into a genetic algorithm pipeline. This pipeline handles:
1. **Selection**: how individuals are chosen for breeding (or carried forward as elites). 
2. **Crossover**: how two individuals exchange parts of their trees.
3. **Mutation**: how individuals are altered (sub-trees replaced, etc.).
The pipeline also needs a **problem** to score the population. Since the combination rule is yours (EvoGP does not know it), the combined route uses the `CustomLoss` problem, in which you write the rule explicitly - see the complete example below.
You can just use the followings:


In [2]:
from evogp.algorithm import (
    DefaultSelection,
    CombinedDefaultMutation,
    CombinedDefaultCrossover,
)
crossover=CombinedDefaultCrossover()
mutation=CombinedDefaultMutation(
    mutation_rate=0.2, descriptors=descriptor
)
selection=DefaultSelection(survival_rate=0.3, elite_rate=0.01)

## 4. Converting a CombinedTree (or CombinedForest) to Sympy Expressions
Once you have evolved a best solution (or a set of solutions), you can:
- Access each parameter sub-tree directly (e.g., `best.A`, `best.B`, `best.C`) - `CombinedTree` sets one attribute per `data_info` key.
- Convert these sub-trees into **sympy expressions** by calling `.to_sympy_expr()`.
- Rebuild the **entire** analytic formula by applying your rule to those expressions, e.g. `formula(best.A.to_sympy_expr(), best.B.to_sympy_expr(), best.C.to_sympy_expr())`. Note that `CombinedTree` itself provides no `to_sympy_expr`, precisely because it does not know your rule.
This is helpful for interpretability (e.g., to see an analytic formula of your best solution).


## Complete Example

In [3]:
# import nessary packages
import torch
from evogp.tree import CombinedForest, GenerateDescriptor
from evogp.algorithm import (
    GeneticProgramming,
    DefaultSelection,
    CombinedDefaultMutation,
    CombinedDefaultCrossover,
)
from evogp.problem import CustomLoss
from evogp.pipeline import StandardPipeline


In [4]:
# Prepare sample XOR data (the tensors live on CUDA: EvoGP's tree containers and
# GenerateDescriptor are CUDA-bound, while the evaluation engine itself is CPU-side by
# default - no custom CUDA kernel is involved any more)
XOR_INPUTS = torch.tensor(
    [
        [0, 0, 0],
        [0, 0, 1],
        [0, 1, 0],
        [0, 1, 1],
        [1, 0, 0],
        [1, 0, 1],
        [1, 1, 0],
        [1, 1, 1],
    ],
    dtype=torch.float,
    device="cuda",
)
XOR_OUTPUTS = torch.tensor(
    [[0], [1], [1], [0], [1], [0], [0], [1]],
    dtype=torch.float,
    device="cuda",
)

# The combined route is driven by named data: `existing_data` holds the fixed data
# (the three XOR input bits plus the target), keyed by the names used in `data_info`.
existing_data = {
    "x0": XOR_INPUTS[:, 0],
    "x1": XOR_INPUTS[:, 1],
    "x2": XOR_INPUTS[:, 2],
    "y": XOR_OUTPUTS[:, 0],
}

# The top-level rule lives here, in the loss we ask the algorithm to minimize.
# `CustomLoss` inspects the signature of the loss: the first `len(existing_data)`
# parameters are fed the fixed data, the remaining ones receive the sub-tree outputs
# (batched over the population). So the signature must list the `existing_data` keys
# first, then the `data_info` keys.
formula = lambda A, B, C: A + B * C


def loss_func(x0, x1, x2, y, A, B, C):
    prediction = formula(A, B, C)
    return ((prediction - y) ** 2).mean(dim=-1)


problem = CustomLoss(existing_data=existing_data, loss_func=loss_func)


In [5]:
# Create a descriptor for generating sub-trees
descriptor = GenerateDescriptor(
    max_tree_len=16,
    input_len=3,                     # every data_info key lists 3 inputs
    output_len=1,                    # each sub-tree is single-output
    using_funcs=["+", "-", "*", "/"],
    max_layer_cnt=4,
    const_samples=[0, 0.5, 1]
)
# Map the sub-trees of the formula `A + B * C` to the inputs they consume
# (the names "x0", "x1", "x2" are the keys of the `existing_data` dict above).
data_info = {
    "A": ["x0", "x1", "x2"],
    "B": ["x0", "x1", "x2"],
    "C": ["x0", "x1", "x2"],
}
# Create the initial CombinedForest: one sub-forest per data_info key.
initial_population = CombinedForest.random_generate(
    pop_size=500,
    data_info=data_info,
    descriptors=descriptor,
)


In [6]:
# Set up our GeneticProgramming algorithm with default operators
algorithm = GeneticProgramming(
    initial_forest=initial_population,
    crossover=CombinedDefaultCrossover(),
    mutation=CombinedDefaultMutation(
        mutation_rate=0.2, 
        descriptors=descriptor.update(max_layer_cnt=3)
    ),
    selection=DefaultSelection(survival_rate=0.3, elite_rate=0.01),
)

In [7]:
# We wrap it in a StandardPipeline for convenience:
pipeline = StandardPipeline(
    algorithm,
    problem,
    generation_limit=20,
)
# Run the pipeline and get the best solution
best = pipeline.run()


Generation: 0, Cost time: 5752.56ms
 	fitness: valid cnt: 69, max: -0.2656, min: -18.5000, mean: -2.2889, std: 2.8923

Generation: 1, Cost time: 2953.14ms
 	fitness: valid cnt: 194, max: -0.2500, min: -21.7266, mean: -2.0715, std: 2.7600

Generation: 2, Cost time: 2312.00ms
 	fitness: valid cnt: 434, max: -0.2500, min: -22.7500, mean: -1.4796, std: 1.8294

Generation: 3, Cost time: 1832.24ms
 	fitness: valid cnt: 450, max: -0.2188, min: -7.5000, mean: -0.8619, std: 0.8828

Generation: 4, Cost time: 1132.21ms
 	fitness: valid cnt: 467, max: -0.1875, min: -7.2500, mean: -0.6709, std: 0.6719

Generation: 5, Cost time: 1119.57ms
 	fitness: valid cnt: 456, max: -0.1875, min: -4.5000, mean: -0.5494, std: 0.4893

Generation: 6, Cost time: 1463.29ms
 	fitness: valid cnt: 461, max: -0.1875, min: -3.5000, mean: -0.4198, std: 0.3734

Generation: 7, Cost time: 946.08ms
 	fitness: valid cnt: 463, max: -0.1875, min: -5.2500, mean: -0.4072, std: 0.4028

Generation: 8, Cost time: 1143.13ms
 	fitness: 

In [8]:
# Evaluate the best solution. The combined containers are dict-driven: named inputs in,
# one output per data_info key (i.e. per sub-tree) out. `to_combined_forest()` lines the
# single best tree up as a population of one, which is what the batched (all trees x all
# data rows) evaluator `batch_forward` expects; the top-level formula is applied on top.
sub_outputs = best.to_combined_forest().batch_forward(
    {
        "x0": XOR_INPUTS[:, 0],
        "x1": XOR_INPUTS[:, 1],
        "x2": XOR_INPUTS[:, 2],
    }
)
pred_res = formula(sub_outputs["A"], sub_outputs["B"], sub_outputs["C"]).reshape(-1)
print("Predictions from best:", pred_res)


Predictions from best: tensor([ 0.5000,  1.0000,  1.2500,  0.2500,  0.5000,  0.0000, -0.2500,  0.5000],
       device='cuda:0')


In [9]:
# Check the best solution:
# Check each parts of the solution:
best.A.to_sympy_expr()  # use "A" because we named it in data_info


0.500000

In [10]:
print(best.B.to_sympy_expr())
print(best.C.to_sympy_expr())

-x0*x1*x2 - x1 + 2*x2
(0.5 - x0)*(-x1 + x2 - 0.5)


In [11]:
# Check the whole solution: the very same `formula` callable also applies to the
# sub-tree expressions (it is plain Python, so it works on sympy objects too).
formula(best.A.to_sympy_expr(), best.B.to_sympy_expr(), best.C.to_sympy_expr())


(0.5 - x0)*(-x1 + x2 - 0.5)*(-x0*x1*x2 - x1 + 2*x2) + 0.5